## Aurora_V4 - kWh

#### <span style="color:red; font-weight:bold"> Instructions for Using this Jupyter Notebook:</span>
This Notebook is using end point value minus start point value of kWh to calculate the kWh within one fiscal year.

<span style="color:royalblue">(0) Install Python packages (one-time setup):</span>
- In terminal, run: **pip3 install scikit-learn**

<span style="color:royalblue">(1) Place **data_clean.py** and this Notebook in the same folder (already done)</span>

<span style="color:royalblue">(2) Place **aurora_v4.meter_info.csv,** **building_complex_master_sheet_fyXX.xlsx**, **special_meters.xlsx**, and the **raw data files** in the same directory (already done)</span> 
- **aurora_v4.meter_info.csv** is exported from the database
- **NOTE:** Filename of the raw data files should include the variable name (i.e., kwh)

<span style="color:royalblue">(3) Modify the **Parameters in Section 1** as needed before running this Notebook:</span> 
- Time range, FY, etc.

<span style="color:royalblue">(4) Run the Notebook:</span> 
- Set **Checked = False** and run it once.
- Check **special_meters_plots_fyXX.pdf** located in **output_dir**.
- If there are any unusual periods in the meter reading data, record them in **special_meters.xlsx** located in **input_dir**.
- If no problem, set **Checked = True** and run the notebook again.

<span style="color:royalblue">**Use R² to define Special Meters (R² < 0.9):**</span> 

- **R² (Coefficient of Determination) Definition:** Measures how closely the meter’s kWh readings follow a perfect linear increase (straight) line over time using linear regression.

- **R² Values and Meaning:**
    - R² ≥ 0.9: Excellent — the meter closely follows the expected linear trend.

    - 0.5 ≤ R² < 0.9: Moderate — the meter shows noticeable deviations; may have some irregular readings.

    - 0 < R² < 0.5: Poor — the meter data is highly irregular.

    - R² = 0: All values missing — no valid data.

    - R² = -1: Missing points at start or end of the fiscal year; scaling applied if enough points exist (> 5 months).

    - R² = -2: Stuck points at start or end of the fiscal year; scaling applied.
 
    - R² = -3: Meter restarts at least 1 time.

### 1. Parameters

In [31]:
############ CHANGE PARAMETERS AS NEEDED #############

Checked = False  # Set to True after you review the special_meters plot.
Insert = False   # Leave False for now unless you want to push results into the master sheet.
# Checked = False#True#   # Set to True after you checked the special_meters plot.
# Insert = False   # Set to False if there's no building_complex_master_sheet_fyXX.xlsx for that fiscal year.

# Time Range: Select One Fiscal Year
start_time = "2025-07-23 09:40:50" #"2024-07-01 00:00:00"  #2025-07-23 09:40:50
end_time = "2025-10-17 11:39:02" #"2025-07-01 00:00:00"
FY = ""
# FY = "_fy25"

######################################################

In [32]:

# Data Directories
input_dir = "../data/extracts/"  # directory for raw data files & other input files

output_dir = "../data/outputs/"  # directory for data outputs (different from input_dir)

plot_dir = "../data/outputs/plots/"  # directory for plot outputs


# Variable
var = 'kwh'


# Input Files
# TODO: file names may change, check with Eileen

var_file = input_dir + "harvest_kwh_" + "250723-251017.csv" # data file 0
meter_info_file = input_dir + "aurora_v4.meter_info.csv"  # contains all meter information
meter_issues_candidates_file = input_dir + "meter_issues_candidates.xlsx"  # auto-generated review file
# TODO: automate create meter issue file
# TODO: rename meter_issues_file to special_meters_file
meter_issues_file = input_dir + "special_meters.xlsx"  # records special meters that need to be corrected

# TODO:meter_info_file added temp above ask Eileen
# meter_info_file = None #input_dir + "harvest.meter_info.csv"  # contains all meter information

# var_file = input_dir + "aurora_v4."+var+".fy22_fy25.csv"  # data file 0
# var_rejects_file = input_dir + "aurora_v4."+var+"_rejects.fy22_fy25.csv"  # data file 1
# meter_info_file = input_dir + "aurora_v4.meter_info.csv"  # contains all meter information
# special_meters_file = input_dir + "aurora_v4." + "special_meters.xlsx"  # records special meters that need to be corrected
# meter_issues_file = input_dir + "meter_issues.xlsx"  # records special meters that need to be corrected

######################################################
if Insert:
    # TODO:
    # commented out for now since about fiscal year sheet
    # insert_sheet = input_dir + "building_complex_master_sheet" + FY + ".xlsx"  # insert output kWh usage into this sheet
    sheet_name = "complex"
    target_col_idx = 10  # insert into column K "Net kWh" (Note: Column A's index = 0)
######################################################

# Output Files
meter_annual_csv = output_dir + "meter_annual_" + var + FY + ".csv"  # annual kwh usage for each meter
building_annual_csv = output_dir + "building_annual_" + var + FY + ".csv"  # annual kwh usage for each building
scaling_detail_csv = output_dir + "meter_scaling_detail" + FY + ".csv"

# Output Figure
special_meters_plot = plot_dir + "special_meters_plots" + FY + ".pdf"

#do i ignore these?
# Exclude: Meters of buildings equipped with PV and Student Health
meters_with_pv = [
    'bachman_hall_main',
    'campus_ctr_main',
    'dance_bldg_main',
    'gartley_hall_main',
    'warrior_rec_ctr_main'
]
meters_excluded = meters_with_pv + ['student_health_main']  # student_health data is in vitality_v5


# Valid Data Min Length
valid_len = 5*30*96  # A meter should have at least 5-months valid data within 1 fiscal year

# Parameters for data cleaning - no need to change for now
r2_threshold = 0.9

# If a meter restarts more than 5 times in a fiscal year, treat it as a Special Meter
restarts_thres = 5

# Data Frequency
# TODO: check
freq = None
# freq = '15min'

# Schema
schema = 'harvest'
# schema = 'aurora_v4'


### 2. Imports

In [33]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Alignment
import math

import data_clean as dc  # import self-defined module


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 3. Load Data

In [34]:
# Read var and var_rejects tables
df0 = pd.read_csv(var_file)
df0["datetime"] = pd.to_datetime(df0["datetime"])

# TODO: commented out bc no rejects file for now
# df1 = pd.read_csv(var_rejects_file)

df0.head()

,datetime,meter_name,meter_reading,is_exact,interpolated
0,2025-07-23 09:40:50,admin_serv_1,1381508.0,False,False
1,2025-07-23 09:41:54,admin_serv_1,1381508.0,False,False
2,2025-07-23 09:42:59,admin_serv_1,1381509.0,False,False
3,2025-07-23 09:44:03,admin_serv_1,1381510.0,False,False
4,2025-07-23 09:45:00,admin_serv_1,1381510.0,True,True


### 4. Data Processing

In [35]:
# Old Aurora rejects file swap logic is not needed right now

# # The meter_reading values of these two meters in the kwh_rejects table are switched, so switch them back
# swap_map = {
#     "hale_aloha_ilima_tower_cafe": "hale_aloha_ilima_tower_main",
#     "hale_aloha_ilima_tower_main": "hale_aloha_ilima_tower_cafe"
# }

# df1['meter_name'] = df1['meter_name'].replace(swap_map)


In [36]:
# no rejects file so combined_df is just the main harvest data
combined_df = df0.copy()

# # Concatenate df0 and df1 vertically
# combined_df = pd.concat([df0, df1], ignore_index=True)

# # Convert 'datetime' to datetime type (if not already)
# combined_df['datetime'] = pd.to_datetime(combined_df['datetime'])

# Sort by meter_name and datetime
combined_df["datetime"] = pd.to_datetime(combined_df["datetime"])
combined_df = combined_df.sort_values(by=['meter_name', 'datetime']).reset_index(drop=True)


In [37]:
print(combined_df.head())

             datetime    meter_name  meter_reading  is_exact  interpolated
0 2025-07-23 09:40:50  admin_serv_1      1381508.0     False         False
1 2025-07-23 09:41:54  admin_serv_1      1381508.0     False         False
2 2025-07-23 09:42:59  admin_serv_1      1381509.0     False         False
3 2025-07-23 09:44:03  admin_serv_1      1381510.0     False         False
4 2025-07-23 09:45:00  admin_serv_1      1381510.0      True          True


In [ ]:
# Pivot table with every meter be one column
pivoted_df = combined_df.pivot(index='datetime', columns='meter_name', values='meter_reading').reset_index()

# Fill missing timestamps
full_df = dc.fill_missing_timestamps(pivoted_df, freq)

full_df.head()


,datetime,admin_serv_1,admin_serv_2_main,ag_engineering_main,ag_engineering_mcc,ag_science_main_1,ag_science_main_2,ag_science_mcc,andrews_amp_main,archtecture_main,...,sherman_main_2,spalding_hall_main,st_john_plant_science_main,stan_sheriff_ctr_main_1,stan_sheriff_ctr_main_2,student_health_main,transportation_srvc_main,univ_high_school_3_main,webster_hall_main,wist_annex_1_main
0,2025-07-23 09:40:50,1381508.0,394674.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-07-24 09:40:50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-07-25 09:40:50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-07-26 09:40:50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-07-27 09:40:50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### <span style="color:royalblue">Filter Meters and Time Range:</span>

In [39]:
### Filter One: Retain only main meters and filter out sub and PV meters ###

# Step 1: Read meter info
meter_info = pd.read_csv(meter_info_file)

# Step 2: Exclude certain meters first
all_meters = meter_info['meter_name'].unique()
non_exc_meters = [m for m in all_meters if m not in meters_excluded]

# Step 3: Get all 'main' meters from non-PV meters
main_meters_no_exc = meter_info[
    (meter_info['end_use'] == 'main') & 
    (meter_info['meter_name'].isin(non_exc_meters))]['meter_name'].unique()

# Step 4: Filter full_df to keep only non-PV 'main' meters (plus datetime)
columns_to_keep = ['datetime'] + list(full_df.columns.intersection(main_meters_no_exc))
filtered_df = full_df[columns_to_keep]

# Set index as datetime
filtered_df.set_index('datetime', inplace=True)


In [40]:
# filtered_df.loc["2024-06-10":"2024-06-17", "cmore_hale_main"].plot()

In [41]:
# filtered_df.loc["2024-06-10":"2024-06-16", "cmore_hale_main"].to_csv("TEST.csv", index=True)

In [42]:
### Filter Two: Retain only the selected analysis-window data ###

data = filtered_df.loc[start_time:end_time, :].copy()
data.index = pd.to_datetime(data.index)

# Initial Data Cleaning: Replace all 0s with NaN in the entire DataFrame 
data = data.replace(0, np.nan)

data.head(2)


,admin_serv_1,admin_serv_2_main,ag_engineering_main,ag_science_main_1,ag_science_main_2,andrews_amp_main,archtecture_main,bachman_hall_annex,biomedical_science_main_a,biomedical_science_main_b,...,sherman_main_1,sherman_main_2,spalding_hall_main,st_john_plant_science_main,stan_sheriff_ctr_main_1,stan_sheriff_ctr_main_2,transportation_srvc_main,univ_high_school_3_main,webster_hall_main,wist_annex_1_main
datetime,,,,,,,,,,,,,,,,,,,,,
2025-07-23 09:40:50,1381508.0,394674.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-24 09:40:50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<!-- ##### <span style="color:royalblue">Correct Special Meters:</span>
- See **meter_issues_file.xlsx** in input_dir for details. -->

##### <span style="color:royalblue">Correct Special Meters:</span>
- If `meter_issues.xlsx` exists, its reviewed corrections are applied first.
- Also auto-generates `meter_issues_candidates.xlsx` after bad meters are detected.

In [43]:
# if os.path.exists(meter_issues_file):
#     data_corrected = dc.apply_special_meter_corrections(data, meter_issues_file)
# else:
#     print("special_meters.xlsx not found, so no manual meter corrections were applied.")
#     data_corrected = data.copy()

data_corrected = dc.apply_special_meter_corrections(data, meter_issues_file)

No special meter file provided. Skipping manual corrections.


##### <span style="color:royalblue">Find Special Meters:</span>

In [44]:
# Find special meters (R² < 0.9) after applying any manual corrections
df_special_meters, df_restarts = dc.find_special_meters(data_corrected, r2_threshold)
df_special_meters
# df_special_meters, df_restarts = dc.find_special_meters(data, r2_threshold)

,meter_name,r2,info
0,admin_serv_1,-1,missing end
1,admin_serv_2_main,-1,missing end
2,ag_engineering_main,-1,missing start and end
3,ag_science_main_2,-1,missing start and end
4,andrews_amp_main,-1,missing start and end
...,...,...,...
67,sherman_main_2,0,all NaN
68,st_john_plant_science_main,0,all NaN
69,stan_sheriff_ctr_main_1,0,all NaN
70,stan_sheriff_ctr_main_2,0,all NaN


##### <span style="color:royalblue">Plot Special Meters with Info:</span>

In [45]:
print("data_corrected shape:", data_corrected.shape)
print("index type:", type(data_corrected.index))
print("index min:", data_corrected.index.min())
print("index max:", data_corrected.index.max())
print("special meters:")
print(df_special_meters.head())

data_corrected shape: (87, 72)
index type: <class 'pandas.DatetimeIndex'>
index min: 2025-07-23 09:40:50
index max: 2025-10-17 09:40:50
special meters:
            meter_name  r2                   info
0         admin_serv_1  -1            missing end
1    admin_serv_2_main  -1            missing end
2  ag_engineering_main  -1  missing start and end
3    ag_science_main_2  -1  missing start and end
4     andrews_amp_main  -1  missing start and end


In [46]:
# Auto-create a candidate meter issues workbook for review
df_meter_issues_candidates = dc.create_special_meters_workbook(
    data_corrected,
    meter_issues_candidates_file,
    df_bad_meters=df_special_meters,
    df_restarts=df_restarts,
    overwrite=True,
)
# TODO: change df_bad_meters to df_special_meters in the above function call
# TODO: ^ after change in the function definition in data_clean.py

df_meter_issues_candidates.head(10)

# Plot special meters
dc.plot_special_meters(data_corrected, df_special_meters, special_meters_plot)
# dc.plot_special_meters(data, df_special_meters, special_meters_plot)

Auto-generated special meters workbook saved to ../data/extracts/meter_issues_candidates.xlsx


/Users/cassiehuber/Documents/GitHub/harvest_kwh_prep/notebooks/data_clean.py:672: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  #     if df_bad_meters is not None and not df_bad_meters.empty:
/Users/cassiehuber/Documents/GitHub/harvest_kwh_prep/notebooks/data_clean.py:672: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  #     if df_bad_meters is not None and not df_bad_meters.empty:
/Users/cassiehuber/Documents/GitHub/harvest_kwh_prep/notebooks/data_clean.py:672: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  #     if df_bad_meters is not None and not df_bad_meters.empty:
/Users/cassiehuber/Documents/GitHub/harvest_kwh_prep/notebooks/data_clean.py:672: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  #     if

TimeoutError: [Errno 60] Operation timed out

In [ ]:
if not Checked:
    raise SystemExit("Execution stopped because Checked = False")


#### <span style="color:red">!!! Note: Check special_meters_plots_fyXX.pdf first before run the following cells !!!</span>

### 5. Calculate (End - Start) Difference for Annual kWh

In [ ]:
### Compute kWh difference between End point and Start point; Scaling applied for some special meters ###

# Step 1: Compute differences
result_df = dc.compute_meter_differences(
    data_corrected,
    start_time,
    end_time,
    df_special_meters,
    df_restarts,
    valid_len=valid_len,
    r2_threshold=r2_threshold,
    restarts_thres=restarts_thres,
)

################ SCALING DETAIL ################
scaling_detail = result_df.reset_index()[[
    "meter_name", "raw_difference", "difference", "R²", "info", "% scaled"
]]

scaling_detail["estimated_kwh"] = (
    scaling_detail["difference"] - scaling_detail["raw_difference"]
)

scaling_detail.to_csv(scaling_detail_csv, index=False)
# scaling_detail.to_csv("../data/outputs/meter_scaling_detail_fy25.csv", index=False)
print("Total estimated kWh:", scaling_detail["estimated_kwh"].sum())
################################################

# export meter-level differences only
df_all = result_df.reset_index()[["meter_name", "difference"]].copy()
df_all.rename(columns={"difference": f"annual_{var}"}, inplace=True)
df_all[f"annual_{var}"] = df_all[f"annual_{var}"].round(1)
df_all.to_csv(meter_annual_csv, index=False)

# skip building-level export for now because there is no meter_info file yet

# # Step 2: Export all meters' differences -> annual kWh usage (CSV, rounded 1 decimal)
# df_all = dc.export_meter_differences(result_df, meter_info_file, meter_annual_csv, var=var)

# # Step 3: Export building-level differences -> annual kWh per building (CSV)
# df_building_sum = dc.export_building_differences(df_all, building_annual_csv, var=var)



##### <span style="color:royalblue">Insert data into master_sheet:</span>

In [ ]:
### Insert kWh difference (annual usage) data into the Master Sheet ####

if Insert:

    # Load workbook and sheet
    wb = load_workbook(insert_sheet)
    ws = wb[sheet_name]

    df_diff = df_building_sum

    # Create mapping: building_complex_name -> annual_{var}
    diff_dict = dict(zip(df_diff['building_complex_name'], df_diff[f'annual_{var}']))

    # Loop rows starting from row 3
    for row in ws.iter_rows(min_row=3):
        building = row[2].value    # Column C (index 2)
        if building in diff_dict:
            val = diff_dict[building]

            # Column K (0-indexed = 10)
            cell = row[target_col_idx]

            # Handle NaN
            if val is None or (isinstance(val, float) and math.isnan(val)):
                cell.value = None
            else:
                cell.value = val

            # Right alignment
            cell.alignment = Alignment(horizontal="right")

    # Save workbook
    wb.save(insert_sheet)
